# Data Ingestion

In [1]:
import os
import pandas as pd

# Load all 10 datasets from data/raw
datasets = {}
raw_data_path = r'C:\Users\Asus\OneDrive\Desktop\Bluestock_intern\Bluestock-project-repo-1\bluestock_mf_capstone\data\raw'

for filename in sorted(os.listdir(raw_data_path)):
    if filename.endswith('.csv'):
        file_path = os.path.join(raw_data_path, filename)
        datasets[filename] = pd.read_csv(file_path)

# Display loaded datasets
print(f"Loaded {len(datasets)} datasets:")
for name in datasets.keys():
    print(f"  - {name}: \n shape:{datasets[name].shape} \n data type:{datasets[name].dtypes.iloc[:]} \n head:{datasets[name].head()}\n")

Loaded 16 datasets:
  - 01_fund_master.csv: 
 shape:(40, 15) 
 data type:amfi_code               int64
fund_house                str
scheme_name               str
category                  str
sub_category              str
plan                      str
launch_date               str
benchmark                 str
expense_ratio_pct     float64
exit_load_pct         float64
min_sip_amount          int64
min_lumpsum_amount      int64
fund_manager              str
risk_category             str
sebi_category_code        str
dtype: object 
 head:   amfi_code       fund_house                                   scheme_name  \
0     119551  SBI Mutual Fund     SBI Bluechip Fund - Regular Plan - Growth   
1     119552  SBI Mutual Fund      SBI Bluechip Fund - Direct Plan - Growth   
2     119598  SBI Mutual Fund    SBI Small Cap Fund - Regular Plan - Growth   
3     119599  SBI Mutual Fund     SBI Small Cap Fund - Direct Plan - Growth   
4     119120  SBI Mutual Fund  SBI Magnum Gilt Fund - Regular

In [2]:
import requests
import pandas as pd
from datetime import datetime
import json

# API configuration
API_BASE_URL = "https://api.mfapi.in/mf"

# Define the 5 key schemes to fetch
schemes = {
    '125497': 'HDFC Top 100 Direct',
    '119551': 'SBI Bluechip',
    '120503': 'ICICI Bluechip',
    '118632': 'Nippon Large Cap',
    '119092': 'Axis Bluechip',
    '120841': 'Kotak Bluechip'
}

nav_data = {}

# Fetch NAV for each scheme
print("Fetching live NAV data from mfapi.in...\n")
for scheme_code, scheme_name in schemes.items():
    try:
        url = f"{API_BASE_URL}/{scheme_code}"
        response = requests.get(url, timeout=10)
        response.raise_for_status()  # Raise error for bad status
        
        data = response.json()
        nav_data[scheme_code] = {
            'name': scheme_name,
            'data': data,
            'status': 'SUCCESS'
        }
        
        # Extract key info
        if 'data' in data and len(data['data']) > 0:
            latest_nav = data['data'][0]
            print(f"  {scheme_name} ({scheme_code})")
            print(f"  Latest NAV: ₹{latest_nav['nav']} (Date: {latest_nav['date']})")
            print(f"  Total records: {len(data['data'])}\n")
        
    except requests.exceptions.RequestException as e:
        print(f"  Error fetching {scheme_name} ({scheme_code}): {str(e)}\n")
        nav_data[scheme_code] = {'status': 'FAILED', 'error': str(e)}

# Save each scheme's NAV data as CSV
raw_data_path = r'C:\Users\Asus\OneDrive\Desktop\Bluestock_intern\Bluestock-project-repo-1\bluestock_mf_capstone\data\raw'

print("\n" + "="*60)
print("Saving NAV data to CSV files...\n")

for scheme_code, info in nav_data.items():
    if info['status'] == 'SUCCESS' and 'data' in info['data']:
        try:
            # Convert to DataFrame
            df = pd.DataFrame(info['data']['data'])
            
            # Save CSV
            csv_filename = f"nav_{scheme_code}_{info['name'].replace(' ', '_')}.csv"
            csv_path = os.path.join(raw_data_path, csv_filename)
            df.to_csv(csv_path, index=False)
            
            print(f"Saved: {csv_filename}")
            print(f"Shape: {df.shape}")
            print(f"Columns: {list(df.columns)}\n")
            
        except Exception as e:
            print(f"Error saving {scheme_code}: {str(e)}\n")

print("="*60)
print("NAV data ingestion complete!")

Fetching live NAV data from mfapi.in...

  HDFC Top 100 Direct (125497)
  Latest NAV: ₹192.31950 (Date: 01-06-2026)
  Total records: 3091

  SBI Bluechip (119551)
  Latest NAV: ₹104.70250 (Date: 01-06-2026)
  Total records: 3236

  ICICI Bluechip (120503)
  Latest NAV: ₹103.09480 (Date: 01-06-2026)
  Total records: 3307

  Nippon Large Cap (118632)
  Latest NAV: ₹97.19440 (Date: 01-06-2026)
  Total records: 3298

  Axis Bluechip (119092)
  Latest NAV: ₹6156.75320 (Date: 01-06-2026)
  Total records: 3565

  Kotak Bluechip (120841)
  Latest NAV: ₹246.98140 (Date: 01-06-2026)
  Total records: 3301


Saving NAV data to CSV files...

Saved: nav_125497_HDFC_Top_100_Direct.csv
Shape: (3091, 2)
Columns: ['date', 'nav']

Saved: nav_119551_SBI_Bluechip.csv
Shape: (3236, 2)
Columns: ['date', 'nav']

Saved: nav_120503_ICICI_Bluechip.csv
Shape: (3307, 2)
Columns: ['date', 'nav']

Saved: nav_118632_Nippon_Large_Cap.csv
Shape: (3298, 2)
Columns: ['date', 'nav']

Saved: nav_119092_Axis_Bluechip.csv
Sh

## Fund Master Exploration

In [3]:
fund_master_path = os.path.join(raw_data_path, '01_fund_master.csv')
if os.path.exists(fund_master_path):
    fund_master = pd.read_csv(fund_master_path)
    print(f"\nFund Master loaded successfully!")
    print(f"Shape: {fund_master.shape}")
    print(f"Columns: {list(fund_master.columns)}\n")
else:
    print(f"⚠ Fund Master file not found at: {fund_master_path}")
    print(f"Available files: {os.listdir(raw_data_path)}")


Fund Master loaded successfully!
Shape: (40, 15)
Columns: ['amfi_code', 'fund_house', 'scheme_name', 'category', 'sub_category', 'plan', 'launch_date', 'benchmark', 'expense_ratio_pct', 'exit_load_pct', 'min_sip_amount', 'min_lumpsum_amount', 'fund_manager', 'risk_category', 'sebi_category_code']



In [4]:
if os.path.exists(fund_master_path):
    print("First 5 records:")
    print(fund_master.head())
    print("\n")
else:
    print(f"⚠ Fund Master file not found at: {fund_master_path}")
    print(f"Available files: {os.listdir(raw_data_path)}")


First 5 records:
   amfi_code       fund_house                                   scheme_name  \
0     119551  SBI Mutual Fund     SBI Bluechip Fund - Regular Plan - Growth   
1     119552  SBI Mutual Fund      SBI Bluechip Fund - Direct Plan - Growth   
2     119598  SBI Mutual Fund    SBI Small Cap Fund - Regular Plan - Growth   
3     119599  SBI Mutual Fund     SBI Small Cap Fund - Direct Plan - Growth   
4     119120  SBI Mutual Fund  SBI Magnum Gilt Fund - Regular Plan - Growth   

  category sub_category     plan launch_date                  benchmark  \
0   Equity    Large Cap  Regular  2006-02-14              NIFTY 100 TRI   
1   Equity    Large Cap   Direct  2013-01-01              NIFTY 100 TRI   
2   Equity    Small Cap  Regular  2009-09-09       BSE 250 SmallCap TRI   
3   Equity    Small Cap   Direct  2013-01-01       BSE 250 SmallCap TRI   
4     Debt         Gilt  Regular  2000-12-30  CRISIL Dynamic Gilt Index   

   expense_ratio_pct  exit_load_pct  min_sip_amount  min_

In [5]:
if os.path.exists(fund_master_path):
    fund_houses = fund_master['fund_house'].unique() if 'fund_house' in fund_master.columns else fund_master['FundHouse'].unique()
    print(f"Total Fund Houses: {len(fund_houses)}")
    print(f"\nList of Fund Houses:")
    for idx, house in enumerate(sorted(fund_houses), 1):
        count = len(fund_master[fund_master['fund_house'] == house]) if 'fund_house' in fund_master.columns else len(fund_master[fund_master['FundHouse'] == house])
        print(f"  {idx:2d}. {house} ({count} schemes)")
else:
    print(f"⚠ Fund Master file not found at: {fund_master_path}")
    print(f"Available files: {os.listdir(raw_data_path)}")

Total Fund Houses: 10

List of Fund Houses:
   1. Aditya Birla Sun Life MF (3 schemes)
   2. Axis Mutual Fund (4 schemes)
   3. DSP Mutual Fund (3 schemes)
   4. HDFC Mutual Fund (5 schemes)
   5. ICICI Prudential MF (5 schemes)
   6. Kotak Mahindra MF (4 schemes)
   7. Mirae Asset MF (3 schemes)
   8. Nippon India MF (5 schemes)
   9. SBI Mutual Fund (5 schemes)
  10. UTI Mutual Fund (3 schemes)


In [6]:
if os.path.exists(fund_master_path):
    if 'category' in fund_master.columns:
        categories = fund_master['category'].unique()
    print(f"Total Categories: {len(categories)}")
    print(f"\nCategories:")
    for idx, cat in enumerate(sorted(categories), 1):
        count = len(fund_master[fund_master['category'] == cat])
        print(f"  {idx}. {cat} ({count} schemes)")
else:
    print(f"⚠ Fund Master file not found at: {fund_master_path}")
    print(f"Available files: {os.listdir(raw_data_path)}")

Total Categories: 2

Categories:
  1. Debt (6 schemes)
  2. Equity (34 schemes)


In [7]:
if os.path.exists(fund_master_path):
    if 'sub_category' in fund_master.columns:
        subcategories = fund_master['sub_category'].unique()
    print(f"Total Sub-Categories: {len(subcategories)}")
    print(f"\nSub-Categories:")
    for idx, subcat in enumerate(sorted(subcategories), 1):
        count = len(fund_master[fund_master['sub_category'] == subcat])
        print(f"  {idx}. {subcat} ({count} schemes)")
else:
    print(f"⚠ Fund Master file not found at: {fund_master_path}")
    print(f"Available files: {os.listdir(raw_data_path)}")

Total Sub-Categories: 12

Sub-Categories:
  1. ELSS (1 schemes)
  2. Flexi Cap (2 schemes)
  3. Gilt (2 schemes)
  4. Index (1 schemes)
  5. Index/ETF (1 schemes)
  6. Large & Mid Cap (1 schemes)
  7. Large Cap (14 schemes)
  8. Liquid (3 schemes)
  9. Mid Cap (7 schemes)
  10. Short Duration (1 schemes)
  11. Small Cap (6 schemes)
  12. Value (1 schemes)


In [8]:
if os.path.exists(fund_master_path):
    if 'risk_category' in fund_master.columns:
        risk_category = fund_master['risk_category'].unique()
    print(f"Total Risk Categories: {len(risk_category)}")
    print(f"\nRisk Categories:")
    for idx, category in enumerate(sorted(risk_category), 1):
        count = len(fund_master[fund_master['risk_category'] == category]) if 'risk_category' in fund_master.columns else len(fund_master[fund_master['RiskGrade'] == category])
        print(f"  {idx}. {category} ({count} schemes)")
else:
    print(f"⚠ Fund Master file not found at: {fund_master_path}")
    print(f"Available files: {os.listdir(raw_data_path)}")

Total Risk Categories: 5

Risk Categories:
  1. High (8 schemes)
  2. Low (6 schemes)
  3. Moderate (16 schemes)
  4. Moderately High (4 schemes)
  5. Very High (6 schemes)


In [9]:
if os.path.exists(fund_master_path):
    if 'risk_category' in fund_master.columns:
        risk_category = fund_master['risk_category'].unique()
    print(f"Total Risk Categories: {len(risk_category)}")
    print(f"\nRisk Categories:")
    for idx, category in enumerate(sorted(risk_category), 1):
        count = len(fund_master[fund_master['risk_category'] == category]) if 'risk_category' in fund_master.columns else len(fund_master[fund_master['RiskGrade'] == category])
        print(f"  {idx}. {category} ({count} schemes)")
else:
    print(f"⚠ Fund Master file not found at: {fund_master_path}")
    print(f"Available files: {os.listdir(raw_data_path)}")

Total Risk Categories: 5

Risk Categories:
  1. High (8 schemes)
  2. Low (6 schemes)
  3. Moderate (16 schemes)
  4. Moderately High (4 schemes)
  5. Very High (6 schemes)


In [12]:
if os.path.exists(fund_master_path):
    scheme_code_col = None
    for col in fund_master.columns:
        if 'scheme_code' in col.lower() or 'code' in col.lower():
            scheme_code_col = col
            break
    
    if scheme_code_col:
        print(f"\nScheme Code Column: {scheme_code_col}")
        print(f"Data Type: {fund_master[scheme_code_col].dtype}")
        print(f"Sample Scheme Codes (first 10):")
        for idx, (code, name) in enumerate(zip(
            fund_master[scheme_code_col].head(10),
            fund_master[[col for col in fund_master.columns if 'scheme_name' in col.lower() or 'name' in col.lower()][0]].head(10)
        ), 1):
            print(f"  {idx:2d}. Code: {code} | Scheme: {name}")
        
        print(f"\nScheme Code Analysis:")
        print(f"  • All numeric: {fund_master[scheme_code_col].dtype in ['int64', 'int32']}")
        print(f"  • Min Code: {fund_master[scheme_code_col].min()}")
        print(f"  • Max Code: {fund_master[scheme_code_col].max()}")
        print(f"  • Total Unique Codes: {fund_master[scheme_code_col].nunique()}")
        print(f"  • Duplicate Codes: {len(fund_master) - fund_master[scheme_code_col].nunique()}")
else:
    print(f"⚠ Fund Master file not found at: {fund_master_path}")
    print(f"Available files: {os.listdir(raw_data_path)}")


Scheme Code Column: amfi_code
Data Type: int64
Sample Scheme Codes (first 10):
   1. Code: 119551 | Scheme: SBI Bluechip Fund - Regular Plan - Growth
   2. Code: 119552 | Scheme: SBI Bluechip Fund - Direct Plan - Growth
   3. Code: 119598 | Scheme: SBI Small Cap Fund - Regular Plan - Growth
   4. Code: 119599 | Scheme: SBI Small Cap Fund - Direct Plan - Growth
   5. Code: 119120 | Scheme: SBI Magnum Gilt Fund - Regular Plan - Growth
   6. Code: 100016 | Scheme: HDFC Top 100 Fund - Regular Plan - Growth
   7. Code: 125497 | Scheme: HDFC Top 100 Fund - Direct Plan - Growth
   8. Code: 100033 | Scheme: HDFC Mid-Cap Opportunities Fund - Regular - Growth
   9. Code: 125498 | Scheme: HDFC Mid-Cap Opportunities Fund - Direct - Growth
  10. Code: 100025 | Scheme: HDFC Short Term Debt Fund - Regular - Growth

Scheme Code Analysis:
  • All numeric: True
  • Min Code: 100016
  • Max Code: 149324
  • Total Unique Codes: 40
  • Duplicate Codes: 0


In [15]:
def summarize_data_quality(df, name):
    n_rows, n_cols = df.shape
    print(f"\n{name}")
    print("-" * len(name))
    print(f"Rows: {n_rows}, Columns: {n_cols}")
    print(f"Duplicate rows: {df.duplicated().sum()}")
    null_counts = df.isna().sum()
    null_columns = null_counts[null_counts > 0].sort_values(ascending=False)
    print(f"Columns with missing values: {len(null_columns)}")
    if len(null_columns):
        print(null_columns.to_string())
    print("\nData types:")
    print(df.dtypes.value_counts().to_string())
    date_cols = [col for col in df.columns if "date" in col.lower()]
    if date_cols:
        print("\nDate-like columns:")
        for col in date_cols:
            sample_values = df[col].dropna().astype(str).head(3).tolist()
            print(f"  - {col}: sample values = {sample_values}")
    unique_counts = df.nunique(dropna=False).sort_values()
    print("\nUnique value counts (lowest 10):")
    print(unique_counts.head(10).to_string())


for dataset_name, dataset_df in datasets.items():
    summarize_data_quality(dataset_df, dataset_name)


01_fund_master.csv
------------------
Rows: 40, Columns: 15
Duplicate rows: 0
Columns with missing values: 0

Data types:
str        10
int64       3
float64     2

Date-like columns:
  - launch_date: sample values = ['2006-02-14', '2013-01-01', '2009-09-09']

Unique value counts (lowest 10):
min_sip_amount         1
category               2
plan                   2
exit_load_pct          3
min_lumpsum_amount     3
risk_category          5
sebi_category_code     9
benchmark             10
fund_house            10
sub_category          12

02_nav_history.csv
------------------
Rows: 46000, Columns: 3
Duplicate rows: 0
Columns with missing values: 0

Data types:
int64      1
str        1
float64    1

Date-like columns:
  - date: sample values = ['2022-01-03', '2022-01-04', '2022-01-05']

Unique value counts (lowest 10):
amfi_code       40
date          1150
nav          45594

03_aum_by_fund_house.csv
------------------------
Rows: 90, Columns: 5
Duplicate rows: 0
Columns with missing 

## COMPREHENSIVE DATA STATISTICS

In [23]:
import warnings
warnings.filterwarnings('ignore')

# 1. Overall Dataset Summary
print("\n1. OVERALL DATASET SUMMARY\n")

print(f"Total Datasets Loaded: {len(datasets)}")
total_rows = sum([df.shape[0] for df in datasets.values()])
total_cols = sum([df.shape[1] for df in datasets.values()])
print(f"Total Rows Across All Datasets: {total_rows:,}")
print(f"Total Columns Across All Datasets: {total_cols}")

# 2. Individual Dataset Statistics
print("\n2. INDIVIDUAL DATASET STATISTICS")

for dataset_name, dataset_df in sorted(datasets.items()):
    try:
        print(f"\n {dataset_name}")
        print(f"   Shape: {dataset_df.shape[0]:,} rows x {dataset_df.shape[1]} columns")
        print(f"   Memory Usage: {dataset_df.memory_usage(deep=True).sum() / 1024 / 1024:.2f} MB")
        print(f"   Columns: {', '.join(dataset_df.columns.tolist())}")
    except Exception as e:
        print(f"   Error processing {dataset_name}: {str(e)}")

# 3. Missing Values Analysis
print("\n3. MISSING VALUES ANALYSIS")

has_missing = False
for dataset_name, dataset_df in sorted(datasets.items()):
    try:
        missing = dataset_df.isnull().sum()
        if missing.sum() > 0:
            has_missing = True
            print(f"\n{dataset_name}:")
            for col, count in missing[missing > 0].items():
                pct = (count / len(dataset_df)) * 100
                print(f"   {col}: {count:,} missing ({pct:.2f}%)")
    except Exception as e:
        print(f"   Error in {dataset_name}: {str(e)}")

if not has_missing:
    print(" ✓ No missing values detected in any dataset!")

# 4. Data Types Summary
print("\n4. DATA TYPES SUMMARY")

dtype_summary = {}
for dataset_df in datasets.values():
    for dtype in dataset_df.dtypes:
        dtype_str = str(dtype)
        dtype_summary[dtype_str] = dtype_summary.get(dtype_str, 0) + 1

for dtype, count in sorted(dtype_summary.items()):
    print(f"   {dtype}: {count} columns")

# 5. Numeric Columns Statistics
print("\n5. NUMERIC COLUMNS STATISTICS")

for dataset_name, dataset_df in sorted(datasets.items()):
    try:
        numeric_df = dataset_df.select_dtypes(include=['number'])
        if len(numeric_df.columns) > 0:
            print(f"\n{dataset_name}:")
            stats = numeric_df.describe()
            print(stats.to_string())
        else:
            print(f"\n{dataset_name}: No numeric columns")
    except Exception as e:
        print(f"   Error in {dataset_name}: {str(e)}")

# 6. Categorical Columns Summary
print("\n6. CATEGORICAL COLUMNS SUMMARY")

for dataset_name, dataset_df in sorted(datasets.items()):
    try:
        categorical_df = dataset_df.select_dtypes(include=['object', 'category'])
        if len(categorical_df.columns) > 0:
            print(f"\n{dataset_name}:")
            for col in categorical_df.columns:
                unique_count = dataset_df[col].nunique()
                print(f"   {col}: {unique_count} unique values")
                if unique_count <= 10:
                    print(f"    Values: {list(dataset_df[col].unique())}")
    except Exception as e:
        print(f"   Error in {dataset_name}: {str(e)}")

# 7. Data Quality Metrics
print("\n7. DATA QUALITY METRICS")

for dataset_name, dataset_df in sorted(datasets.items()):
    try:
        completeness = ((dataset_df.shape[0] * dataset_df.shape[1] - dataset_df.isnull().sum().sum()) / 
                       (dataset_df.shape[0] * dataset_df.shape[1])) * 100
        duplicates = dataset_df.duplicated().sum()
        print(f"\n{dataset_name}:")
        print(f"   Completeness: {completeness:.2f}%")
        print(f"   Duplicate Rows: {duplicates}")
        print(f"   Unique Rows: {dataset_df.shape[0] - duplicates}")
    except Exception as e:
        print(f"   Error in {dataset_name}: {str(e)}")

# 8. Fund Master Specific Statistics (if available)
try:
    if 'fund_master' in locals():
        print("\n8. FUND MASTER DETAILED STATISTICS")
        print(f"\nTotal Funds: {len(fund_master)}")
        
        fund_house_col = 'fund_house' if 'fund_house' in fund_master.columns else 'FundHouse'
        print(f"\nFund House Distribution:")
        print(fund_master[fund_house_col].value_counts().to_string())
        
        cat_col = 'category' if 'category' in fund_master.columns else 'Category'
        print(f"\nCategory Distribution:")
        print(fund_master[cat_col].value_counts().to_string())
except Exception as e:
    print(f"Error in Fund Master statistics: {str(e)}")


print(" DATA STATISTICS COMPLETE")




1. OVERALL DATASET SUMMARY

Total Datasets Loaded: 16
Total Rows Across All Datasets: 107,331
Total Columns Across All Datasets: 93

2. INDIVIDUAL DATASET STATISTICS

 01_fund_master.csv
   Shape: 40 rows x 15 columns
   Memory Usage: 0.03 MB
   Columns: amfi_code, fund_house, scheme_name, category, sub_category, plan, launch_date, benchmark, expense_ratio_pct, exit_load_pct, min_sip_amount, min_lumpsum_amount, fund_manager, risk_category, sebi_category_code

 02_nav_history.csv
   Shape: 46,000 rows x 3 columns
   Memory Usage: 3.29 MB
   Columns: amfi_code, date, nav

 03_aum_by_fund_house.csv
   Shape: 90 rows x 5 columns
   Memory Usage: 0.01 MB
   Columns: date, fund_house, aum_lakh_crore, aum_crore, num_schemes

 04_monthly_sip_inflows.csv
   Shape: 48 rows x 6 columns
   Memory Usage: 0.00 MB
   Columns: month, sip_inflow_crore, active_sip_accounts_crore, new_sip_accounts_lakh, sip_aum_lakh_crore, yoy_growth_pct

 05_category_inflows.csv
   Shape: 144 rows x 3 columns
   Memory